# Classify spikes using DTW

In [ ]:
CUT_PARAMS = {
    'signal_name': 'wideband',
    'event_name': 'spk',
    'cut_window': [-0.001, 0.002],
    'fs': 40000
}
SMOOTH_PARAMS = {
    'window_length': 9,
    'polyorder': 3
}
PROCESS_PARAMS = {
    'smooth_params': SMOOTH_PARAMS,
    'dv_threshold': 0.1
}

In [ ]:
INVITRO_DATA_DIR = 'E:/Data/BogiSejtek/20251007/data/invitro/juxta'
INVIVO_DATA_DIR = 'E:/Data/BogiSejtek/20251007/data/invivo'
INVIVO_FEATURE_DIR = 'E:/Data/BogiSejtek/20251007/data/invivo/Preprocessed/SpikeFeatures'


In [ ]:
training_cell_names = [
    '2406241PZ_01',
    '2406242PZ_03',
    '2406263PZ_01',
    '2406282PZ_01',
    '2407031PZ_01',
    '2407031SI_gr2',
    '2407041SI_gr1'
]

In [ ]:
from classifiy_spikes_using_dtw import load_training_data
training_data = load_training_data(
    training_cell_names,
    INVITRO_DATA_DIR,
    cut_params=CUT_PARAMS,
    proc_params=PROCESS_PARAMS
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from classifiy_spikes_using_dtw import mean_shapes_rising

cell_means = []
for i, cell_shapes in enumerate(training_data):
    print(i)
    cell_means.append((
        mean_shapes_rising(
            cell_shapes[0], start_ratio=0.15, end_ratio=0.85
        ),
        mean_shapes_rising(
            cell_shapes[1], start_ratio=0.15, end_ratio=0.85
        )
    ))


In [ ]:
from notebooks.classifiy_spikes_using_dtw import get_rising_phase


fig, axes = plt.subplots(len(training_data) * 2, 2, figsize=(6, 3 * len(training_data)))
for i, proc_shapes in enumerate(training_data):
    for j, (shapes, color) in enumerate(zip(proc_shapes, ['blue', 'orange'])):
        rising_shapes = [
             get_rising_phase(s, 0.15, 0.85, keep_size=True)
             for s in shapes
        ]
        for shape in rising_shapes:
            ax = axes[i * 2][j]
            ax.plot(
                shape.times,
                shape.values,
                color='gray',
                alpha=0.3,
                linewidth=1
            )
            ax = axes[i * 2 + 1][j]
            ax.plot(
                shape.values[:-1],
                np.diff(shape.values),
                color='gray',
                alpha=0.3,
                linewidth=1
            )
        v_mean = np.nanmean([s.values for s in rising_shapes], axis=0)
        axes[i * 2][j].plot(
            rising_shapes[0].times,
            v_mean,
            color=color,
            linewidth=2
        )
        axes[i * 2 + 1][j].plot(
            v_mean[:-1],
            np.diff(v_mean),
            color=color,
            linewidth=2
        )
plt.show()

In [ ]:
pool_spikes = [
    (shape, label)
    for cell_shapes in training_data
    for label, shapes in enumerate(cell_shapes)
    for shape in shapes
]
pool_shapes = [shape for shape, _ in pool_spikes]
y = np.array([label for _, label in pool_spikes])

In [ ]:
from tqdm import tqdm
from dtw import dtw

from .transformers import get_rising_phase

D = np.zeros((len(pool_spikes), len(pool_spikes)))
for i, shape_1 in tqdm(enumerate(pool_shapes), total=len(pool_shapes)):
    for j, shape_2 in enumerate(pool_shapes):
        if i == j:
            D[i][j] = 0
            continue
        phase_1 = np.stack([
            get_rising_phase(shape_1, 0.15, 0.85).values[:-1],
            np.diff(get_rising_phase(shape_1, 0.15, 0.85).values)
        ], axis=1)
        phase_2 = np.stack([
            get_rising_phase(shape_2, 0.15, 0.85).values[:-1],
            np.diff(get_rising_phase(shape_2, 0.15, 0.85).values)
        ], axis=1)

        alignment = dtw(
            phase_1,
            phase_2,
            dist_method='euclidean'
        )
        D[i][j] = getattr(alignment, 'distance')
        D[j][i] = D[i][j]

In [ ]:
sort_idx = np.argsort(y)

In [ ]:
import seaborn as sns
sns.heatmap(D[sort_idx][:, sort_idx])
plt.show()

In [ ]:
import heapq
k = 5
pred_y = []
pred_proba = []
mean_dist = []
for i in range(len(pool_spikes)):
    D_i = D[i]
    D_i[i] = 0
    neighbors = [(D_i[j], y[j]) for j in range(len(pool_spikes)) if j != i]
    neighbors = heapq.nsmallest(k, neighbors)
    neighbor_labels = [label for _, label in neighbors]
    pred_proba.append(np.mean(neighbor_labels))
    mean_dist.append(np.mean([dist for dist, _ in neighbors]))
    pred_y.append(int(np.round(pred_proba[-1])))
pred_y = np.array(pred_y)
pred_proba = np.array(pred_proba)
miss_id = np.where(pred_y != y)[0]


In [ ]:
np.mean(pred_y == y)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
for label, color in zip([0, 1], ['blue', 'orange']):
    sns.scatterplot(
        x=np.where(y == label)[0],
        y=pred_proba[y == label],
        color=color,
        ax=ax
    )
plt.show()

In [ ]:
import umap
reducer = umap.UMAP(
    n_components=2,
    random_state=42,
    metric='precomputed',
    min_dist=0.5,
)
X_reduced = reducer.fit_transform(D)
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
sns.scatterplot(
    x=X_reduced[:, 0],
    y=X_reduced[:, 1],
    hue=y,
    palette=['blue', 'orange'],
    ax=ax
)
sns.scatterplot(
    x=X_reduced[miss_id, 0],
    y=X_reduced[miss_id, 1],
    color='black',
    ax=ax
)
plt.show()

In [ ]:
TEST_FEATURE_PATH = 'E:/Data/BogiSejtek/20251007/data/invivo/Preprocessed/SpikeFeatures/RSI_L2-3'
TEST_CELL_PATH = 'E:/Data/BogiSejtek/20251007/data/invivo/RSI_L2-3'
TEST_CELL_NAME = 'B18_d3t1c1'

In [ ]:
from classifiy_spikes_using_dtw import load_test_data
test_data = load_test_data(
    TEST_CELL_NAME,
    TEST_CELL_PATH,
    TEST_FEATURE_PATH,
    cut_params=CUT_PARAMS,
)
from classifiy_spikes_using_dtw import preprocess_shapes
training_data = preprocess_shapes(test_data, SMOOTH_PARAMS, 0.1)

In [ ]:
import pandas as pd
predictions = []
for test_shape in training_data:
    test_rising_part = get_rising_phase(test_shape, 0.15, 0.85)
    test_spike_phase = np.stack([
        test_rising_part.values[:-1],
        np.diff(test_rising_part.values)
    ], axis=1)

    neighbors = []
    for train_shape, train_y in pool_spikes:
        train_rising_part = get_rising_phase(train_shape, 0.15, 0.85)
        train_spike_phase = np.stack([
            train_rising_part.values[:-1],
            np.diff(train_rising_part.values)
        ], axis=1)

        alignment = dtw(
            test_spike_phase,
            train_spike_phase,
            dist_method='euclidean'
        )
        distance = getattr(alignment, 'distance')
        heapq.heappush(neighbors, (distance, train_y))
        if len(neighbors) > k:
            heapq.heappop(neighbors)

    neighbor_labels = [label for _, label in neighbors]
    pred_proba = np.mean(neighbor_labels)
    predictions.append({
        'y_pred': int(np.round(pred_proba)),
        'proba': pred_proba,
    })
prediction_df = pd.DataFrame(predictions)
prediction_df['y_pred'].value_counts()